# LendIQ — Credit Risk Analytics Pipeline
## Notebook 1: Data Ingestion & ETL

### What this notebook does
This notebook handles the **Extract, Transform, Load (ETL)** phase of the LendIQ pipeline.

- **Extract:** Load raw Lending Club loan data from a flat CSV file (350MB+, 2M+ rows)
- **Transform:** Clean mixed data types, parse dates, handle nulls, and split the 
  monolithic CSV into normalised relational tables that reflect real banking data architecture
- **Load:** Push the cleaned, structured tables into a local PostgreSQL database 
  for SQL-based analytics in the next phase

This is considered the data engineering foundation of the project. Everything in the analytics and modelling phases depends on the quality of what gets built here.

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, URL

In [2]:
df = pd.read_csv(r"C:\Users\sajja\Desktop\SQL_Loan\Data\loan.csv", low_memory=False) 

In [3]:
df.shape

(2260668, 145)

In [4]:
df.describe()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,url,dti,...,deferral_term,hardship_amount,hardship_length,hardship_dpd,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,settlement_amount,settlement_percentage,settlement_term
count,0.0,0.0,2.260668e+06,2.260668e+06,2.260668e+06,2.260668e+06,2.260668e+06,2.260664e+06,0.0,2.258957e+06,...,10613.0,10613.000000,10613.0,10613.000000,8426.000000,10613.000000,10613.000000,33056.000000,33056.000000,33056.000000
mean,NaN,NaN,1.504693e+04,1.504166e+04,1.502344e+04,1.309291e+01,4.458076e+02,7.799243e+04,NaN,1.882420e+01,...,3.0,155.006696,3.0,13.686422,454.840802,11628.036442,193.606331,5030.606922,47.775600,13.148596
std,NaN,NaN,9.190245e+03,9.188413e+03,9.192332e+03,4.832114e+00,2.671737e+02,1.126962e+05,NaN,1.418333e+01,...,0.0,129.113137,0.0,9.728138,375.830737,7615.161123,198.694368,3692.027842,7.336379,8.192319
min,NaN,NaN,5.000000e+02,5.000000e+02,0.000000e+00,5.310000e+00,4.930000e+00,0.000000e+00,NaN,-1.000000e+00,...,3.0,0.640000,3.0,0.000000,1.920000,55.730000,0.010000,44.210000,0.200000,0.000000
25%,NaN,NaN,8.000000e+03,8.000000e+03,8.000000e+03,9.490000e+00,2.516500e+02,4.600000e+04,NaN,1.189000e+01,...,3.0,59.370000,3.0,5.000000,174.967500,5628.730000,43.780000,2227.000000,45.000000,6.000000
50%,NaN,NaN,1.290000e+04,1.287500e+04,1.280000e+04,1.262000e+01,3.779900e+02,6.500000e+04,NaN,1.784000e+01,...,3.0,119.040000,3.0,15.000000,352.605000,10044.220000,132.890000,4172.855000,45.000000,14.000000
75%,NaN,NaN,2.000000e+04,2.000000e+04,2.000000e+04,1.599000e+01,5.933200e+02,9.300000e+04,NaN,2.449000e+01,...,3.0,213.260000,3.0,22.000000,622.792500,16114.940000,284.180000,6870.782500,50.000000,18.000000
max,NaN,NaN,4.000000e+04,4.000000e+04,4.000000e+04,3.099000e+01,1.719830e+03,1.100000e+08,NaN,9.990000e+02,...,3.0,943.940000,3.0,37.000000,2680.890000,40306.410000,1407.860000,33601.000000,521.350000,181.000000


In [5]:
df.head(5)

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,NaN,NaN,2500,2500,2500.0,36 months,13.56,84.92,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,30000,30000,30000.0,60 months,18.94,777.23,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,5000,5000,5000.0,36 months,17.97,180.69,D,D1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,4000,4000,4000.0,36 months,18.94,146.51,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,30000,30000,30000.0,60 months,16.14,731.78,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.info

<bound method DataFrame.info of          id  member_id  loan_amnt  funded_amnt  funded_amnt_inv        term  \
0       NaN        NaN       2500         2500           2500.0   36 months   
1       NaN        NaN      30000        30000          30000.0   60 months   
2       NaN        NaN       5000         5000           5000.0   36 months   
3       NaN        NaN       4000         4000           4000.0   36 months   
4       NaN        NaN      30000        30000          30000.0   60 months   
...      ..        ...        ...          ...              ...         ...   
2260663 NaN        NaN      12000        12000          12000.0   60 months   
2260664 NaN        NaN      12000        12000          12000.0   60 months   
2260665 NaN        NaN      10000        10000          10000.0   36 months   
2260666 NaN        NaN      12000        12000          12000.0   60 months   
2260667 NaN        NaN      16550        16550          16550.0   60 months   

         int_rate  

In [7]:
summary = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.notnull().sum(),
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
print(summary.to_string())

                                              dtype  non_null  null_count  null_pct
id                                          float64         0     2260668    100.00
member_id                                   float64         0     2260668    100.00
loan_amnt                                     int64   2260668           0      0.00
funded_amnt                                   int64   2260668           0      0.00
funded_amnt_inv                             float64   2260668           0      0.00
term                                            str   2260668           0      0.00
int_rate                                    float64   2260668           0      0.00
installment                                 float64   2260668           0      0.00
grade                                           str   2260668           0      0.00
sub_grade                                       str   2260668           0      0.00
emp_title                                       str   2093699      166969   

The raw Lending Club dataset contains 2,260,668 loan records across 145 columns, 
spanning originations from 2007 to 2018. While rich in detail, the dataset reflects 
the messiness of real-world financial data

`id`, `member_id`, and `url` contain zero values across all 2.26M rows. 
These were likely anonymised or deprecated by Lending Club before export. 
They are dropped immediately as they carry no analytical value.

Columns relating to joint applications (`annual_inc_joint`, `dti_joint`, 
`verification_status_joint`, all `sec_app_*` fields) are over 94% null, 
reflecting the fact that the vast majority of loans are individual rather 
than joint applications. Similarly, `hardship_` and `settlement_` columns 
exceed 98% nulls. These capture rare edge-case events and would introduce 
more noise than signal. All are dropped.

`mths_since_last_delinq` (51% null), `next_pymnt_d` (58% null), 
`mths_since_last_record` (84% null), and related derogatory mark timing 
columns are dropped. Their missingness is itself informative (absence of a 
delinquency means no delinquency occurred), but this signal is already 
captured by other delinquency indicator columns that are fully populated.

### Normalisation Strategy
Rather than loading the flat CSV directly into a single table — which would 
be analytically unwieldy; the cleaned dataset will be split into four normalised tables reflecting the natural 
structure of a retail lending book:

- **`borrowers`** — who the person is (demographics, employment, location)
- **`loans`** — what they borrowed (amount, grade, rate, purpose, status)
- **`credit_profile`** — their creditworthiness at origination (DTI, delinquencies, utilisation)
- **`payments`** — what happened after origination (repayment behaviour, recoveries)

In [8]:
# Instead of hardcoding the column names, columns are dropped on a threshold basis 
# (100% missing and above 50% missing, this would drop all columns as discussed above)

# Calculate null percentage for each column
null_pct = (df.isnull().sum() / len(df) * 100)

In [9]:
# Drop columns based on null thresholds
drop_100 = null_pct[null_pct == 100].index.tolist()
drop_above_50 = null_pct[null_pct > 50].index.tolist()
cols_to_drop = list(set(drop_100 + drop_above_50)) # colums to drop

df.drop(columns=cols_to_drop, inplace=True)

In [10]:
print(f"Dropped {len(cols_to_drop)} columns")

Dropped 44 columns


In [11]:
print(f"Columns: {df.shape[1]}") # columns
print(f"Rows: {df.shape[0]}") # rows

Columns: 101
Rows: 2260668


## Checking data types

In [12]:
print(df.dtypes.value_counts())

float64    75
str        22
int64       4
Name: count, dtype: int64


In [13]:
print(df.dtypes)

loan_amnt                       int64
funded_amnt                     int64
funded_amnt_inv               float64
term                              str
int_rate                      float64
                               ...   
total_bc_limit                float64
total_il_high_credit_limit    float64
hardship_flag                     str
disbursement_method               str
debt_settlement_flag              str
Length: 101, dtype: object


In [14]:
pd.set_option('display.max_rows', None)
print(df.dtypes)

loan_amnt                       int64
funded_amnt                     int64
funded_amnt_inv               float64
term                              str
int_rate                      float64
installment                   float64
grade                             str
sub_grade                         str
emp_title                         str
emp_length                        str
home_ownership                    str
annual_inc                    float64
verification_status               str
issue_d                           str
loan_status                       str
pymnt_plan                        str
purpose                           str
title                             str
zip_code                          str
addr_state                        str
dti                           float64
delinq_2yrs                   float64
earliest_cr_line                  str
inq_last_6mths                float64
open_acc                      float64
pub_rec                       float64
revol_bal   

Dates are stored as string and need to be converted.

In [15]:
date_cols = ['issue_d', 'earliest_cr_line', 'last_pymnt_d', 'last_credit_pull_d']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%b-%Y', errors='coerce')

In [16]:
df['term'] = df['term'].str.extract(r'(\d+)').astype(float)

In [18]:
# fix emp_length
emp_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3,
    '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7,
    '8 years': 8, '9 years': 9, '10+ years': 10
}
df['emp_length'] = df['emp_length'].map(emp_map)

In [20]:
print(df[['issue_d', 'term', 'emp_length']].head())

     issue_d  term  emp_length
0 2018-12-01  36.0        10.0
1 2018-12-01  60.0        10.0
2 2018-12-01  36.0         6.0
3 2018-12-01  36.0        10.0
4 2018-12-01  60.0        10.0


In [21]:
df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,hardship_flag,disbursement_method,debt_settlement_flag
0,2500,2500,2500.0,36.0,13.56,84.92,C,C1,Chef,10.0,...,0.0,1.0,0.0,60124.0,16901.0,36500.0,18124.0,N,Cash,N
1,30000,30000,30000.0,60.0,18.94,777.23,D,D2,Postmaster,10.0,...,0.0,1.0,0.0,372872.0,99468.0,15000.0,94072.0,N,Cash,N
2,5000,5000,5000.0,36.0,17.97,180.69,D,D1,Administrative,6.0,...,0.0,0.0,0.0,136927.0,11749.0,13800.0,10000.0,N,Cash,N
3,4000,4000,4000.0,36.0,18.94,146.51,D,D2,IT Supervisor,10.0,...,100.0,0.0,0.0,385183.0,36151.0,5000.0,44984.0,N,Cash,N
4,30000,30000,30000.0,60.0,16.14,731.78,C,C4,Mechanic,10.0,...,0.0,0.0,0.0,157548.0,29674.0,9300.0,32332.0,N,Cash,N


In [22]:
df.columns.tolist()

['loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'inq_last_6mths',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'last_credit_pull_d',
 'collections_12_mths_ex_med',
 'policy_code',
 'application_type',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_bal',
 'open_acc_6m',
 'open_act_il',
 'open_il_12m',
 'open_il_24m',
 'mths_since_rcnt_il',
 'total_bal_il',
 'il_util',
 'open_rv_12m',
 'open_rv_24m',
 'max_bal_bc',
 'all_util',
 'total_rev_hi_lim',
 'i

# Dataset split into databases

In [24]:
# we first create an id which will serve as the primary key linking the different databases together. This is a common practice in data warehousing 
# to ensure that we can easily join tables on a unique identifier. The original column of the dataset was null.
df = df.reset_index(drop=True)
df['loan_id'] = df.index + 1

In [27]:
# Borrowers (contain employment title, duration, home ownership, income, verification status, state, zip code, and credit history)
borrowers = df[[
    'loan_id', 'emp_title', 'emp_length', 'home_ownership', 
    'annual_inc', 'verification_status', 'addr_state', 
    'zip_code', 'earliest_cr_line'
]].copy()

In [28]:
# Loans (contain loan amount, funded amount, term, interest rate, installment, grade, sub-grade, purpose, issue date, loan status, payment plan, application type, disbursement method, and initial list status)
loans = df[[
    'loan_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 
    'term', 'int_rate', 'installment', 'grade', 'sub_grade', 
    'purpose', 'issue_d', 'loan_status', 'pymnt_plan', 
    'application_type', 'disbursement_method', 'initial_list_status'
]].copy()

In [29]:
# Credit Profile (contain debt-to-income ratio, delinquency history, credit utilization, and other credit-related metrics)
credit_profile = df[[
    'loan_id', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 
    'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'acc_now_delinq', 
    'pub_rec_bankruptcies', 'tax_liens', 'mort_acc', 'tot_coll_amt', 
    'tot_cur_bal', 'total_rev_hi_lim', 'bc_util', 'all_util', 'il_util'
]].copy()

In [30]:
# Payments (contain total payment, total principal received, total interest received, total late fees received, recoveries, collection recovery fee, outstanding principal, last payment date, last payment amount, hardship flag, and debt settlement flag)
payments = df[[
    'loan_id', 'total_pymnt', 'total_rec_prncp', 'total_rec_int', 
    'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 
    'out_prncp', 'last_pymnt_d', 'last_pymnt_amnt', 
    'hardship_flag', 'debt_settlement_flag'
]].copy()

In [32]:
# Check
print(f"borrowers:      {borrowers.shape}")
print(f"loans:          {loans.shape}")
print(f"credit_profile: {credit_profile.shape}")
print(f"payments:       {payments.shape}")

borrowers:      (2260668, 9)
loans:          (2260668, 16)
credit_profile: (2260668, 19)
payments:       (2260668, 12)


In [52]:
# connecting to lendiq database
# engine = create_engine('postgresql://postgres:Syzygy009@@localhost:5432/lendiq') # password is causing an issue
connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password="Syzygy009@",
    host="127.0.0.1",
    port=5432,
    database="lendiq"
)

In [53]:
engine = create_engine(connection_url)

In [54]:
with engine.connect() as conn:
    print("Connected successfully!")

Connected successfully!


In [55]:
borrowers.to_sql('borrowers', engine, if_exists='replace', index=False)

668

In [56]:
loans.to_sql('loans', engine, if_exists='replace', index=False)

668

In [57]:
credit_profile.to_sql('credit_profile', engine, if_exists='replace', index=False)

668

In [58]:
payments.to_sql('payments', engine, if_exists='replace', index=False)

668